## 1. Imports y Configuración

In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import os

# Configuración
VIDEO_PATH = "VideosAnalisis\\clip 1 ‐ Hecho con Clipchamp.mp4"
MAPA_PATH = "beachvolleyballcourt.png"
MODEL_PATH = "yolo11n.pt"

# Margen de detección (10% extra fuera del campo)
MARGIN_PERCENT = 0.10

print("✓ Librerías cargadas")

✓ Librerías cargadas


## 2. Funciones Auxiliares

In [3]:
def get_points(event, x, y, flags, params):
    """
    Callback para seleccionar puntos con el mouse.
    Permite marcar hasta max_points puntos en la imagen.
    """
    points = params["points"]
    image = params["image"]
    wname = params["wname"]
    max_points = params["max_points"]

    if event == cv2.EVENT_LBUTTONDOWN and len(points) < max_points:
        points.append([x, y])
        cv2.circle(image, (x, y), 6, (0, 0, 255), -1)
        cv2.putText(image, str(len(points)), (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        cv2.imshow(wname, image)

        if len(points) == max_points:
            cv2.waitKey(300)
            cv2.destroyWindow(wname)


def point_in_polygon_with_margin(point, polygon, margin_percent=0.10):
    """
    Verifica si un punto está dentro de un polígono expandido.
    El polígono se expande margin_percent hacia afuera (excepto el borde inferior).
    
    Args:
        point: (x, y) coordenadas del punto
        polygon: array de puntos que definen el polígono del campo
        margin_percent: porcentaje de expansión (0.10 = 10%)
    
    Returns:
        True si el punto está dentro del área expandida
    """
    # Calcular centro del polígono
    center = np.mean(polygon, axis=0)
    
    # Expandir polígono (excepto borde inferior)
    expanded_polygon = []
    max_y = np.max(polygon[:, 1])  # Borde inferior (Y máxima)
    
    for pt in polygon:
        # Si el punto está en el borde inferior, no expandir hacia abajo
        if abs(pt[1] - max_y) < 5:  # Tolerancia de 5 píxeles
            # Solo expandir lateralmente
            direction = pt - center
            direction[1] = min(0, direction[1])  # No expandir hacia abajo
            expanded_pt = pt + direction * margin_percent
        else:
            # Expandir normalmente
            direction = pt - center
            expanded_pt = pt + direction * margin_percent
        
        expanded_polygon.append(expanded_pt)
    
    expanded_polygon = np.array(expanded_polygon, dtype=np.int32)
    
    # Verificar si el punto está dentro del polígono expandido
    result = cv2.pointPolygonTest(expanded_polygon, point, False)
    return result >= 0


print("✓ Funciones auxiliares definidas")

✓ Funciones auxiliares definidas


## 3. Cargar Video y Mapa

In [4]:
# Cargar video
video = cv2.VideoCapture(VIDEO_PATH)
if not video.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

# Obtener información del video
fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"✓ Video cargado: {width}x{height}, {fps:.1f} FPS, {total_frames} frames")

# Leer primer frame
ret, first_frame = video.read()
if not ret:
    raise RuntimeError("No se pudo leer el primer frame")

# Cargar mapa
mapa = cv2.imread(MAPA_PATH)
if mapa is None:
    raise FileNotFoundError(f"No se pudo cargar el mapa: {MAPA_PATH}")

print(f"✓ Mapa cargado: {mapa.shape[1]}x{mapa.shape[0]}")

✓ Video cargado: 1920x1080, 30.0 FPS, 481 frames
✓ Mapa cargado: 1536x1024


## 4. Selección de Puntos del Campo

Marca las 4 esquinas del campo en el video (en orden: arriba-izq, arriba-der, abajo-der, abajo-izq).

In [5]:
# Seleccionar puntos del campo en el primer frame
puntos_campo = []
N = 4  # 4 esquinas del campo

imgA = first_frame.copy()
cv2.namedWindow("Selecciona 4 esquinas del campo", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona 4 esquinas del campo", 1200, 800)
cv2.imshow("Selecciona 4 esquinas del campo", imgA)

cv2.setMouseCallback(
    "Selecciona 4 esquinas del campo",
    get_points,
    {"points": puntos_campo, "image": imgA, 
     "wname": "Selecciona 4 esquinas del campo", "max_points": N}
)

print("Marca las 4 esquinas del campo (arriba-izq, arriba-der, abajo-der, abajo-izq)")
cv2.waitKey(0)
cv2.destroyAllWindows()

puntos_campo = np.array(puntos_campo, dtype=np.float32)
print(f"✓ {len(puntos_campo)} puntos seleccionados")
print(f"  Puntos: {puntos_campo.tolist()}")

Marca las 4 esquinas del campo (arriba-izq, arriba-der, abajo-der, abajo-izq)
✓ 4 puntos seleccionados
  Puntos: [[381.0, 714.0], [1582.0, 703.0], [1827.0, 915.0], [130.0, 934.0]]


## 5. Cargar Modelo YOLO

Cargamos el modelo YOLOv11 para detección de personas.

In [8]:
# Cargar modelo YOLO
model = YOLO(MODEL_PATH)
print("✓ Modelo YOLO cargado")

✓ Modelo YOLO cargado


## 6. Tracking Offline en Todo el Video

Procesamos todo el video frame por frame, detectando y trackeando personas.
Filtramos solo las detecciones dentro del campo (con margen del 10%).

In [9]:
# Reiniciar video al inicio
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

# Estructura para almacenar tracking de todos los frames
# tracking_data[frame_idx] = [(track_id, x1, y1, x2, y2, cx, cy), ...]
tracking_data = {}

print(f"Procesando {total_frames} frames...")
print("Esto puede tardar unos minutos...\n")

frame_idx = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    # Realizar detección y tracking con YOLO
    # persist=True mantiene los IDs entre frames
    results = model.track(frame, persist=True, verbose=False, classes=[0])  # 0 = persona
    
    frame_detections = []
    
    for r in results:
        if r.boxes.id is None:
            continue
            
        for box, track_id in zip(r.boxes, r.boxes.id):
            # Coordenadas de la bounding box
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Punto de referencia: centro-abajo (pies del jugador)
            cx = (x1 + x2) // 2
            cy = y2
            
            # Verificar si el punto está dentro del campo (con margen)
            if point_in_polygon_with_margin((cx, cy), puntos_campo, MARGIN_PERCENT):
                frame_detections.append((
                    int(track_id.item()),  # ID del track
                    x1, y1, x2, y2,        # Bounding box
                    cx, cy                 # Centro-abajo
                ))
    
    tracking_data[frame_idx] = frame_detections
    
    # Mostrar progreso cada 50 frames
    if frame_idx % 50 == 0:
        progress = (frame_idx / total_frames) * 100
        print(f"  Frame {frame_idx}/{total_frames} ({progress:.1f}%) - {len(frame_detections)} jugadores detectados")
    
    frame_idx += 1

print(f"\n✓ Tracking completado: {len(tracking_data)} frames procesados")

# Estadísticas
total_detections = sum(len(dets) for dets in tracking_data.values())
avg_detections = total_detections / len(tracking_data) if tracking_data else 0

all_track_ids = set()
for dets in tracking_data.values():
    for det in dets:
        all_track_ids.add(det[0])

print(f"  Total detecciones: {total_detections}")
print(f"  Promedio por frame: {avg_detections:.1f}")
print(f"  IDs únicos detectados: {len(all_track_ids)}")
print(f"  IDs: {sorted(all_track_ids)}")

Procesando 481 frames...
Esto puede tardar unos minutos...

  Frame 0/481 (0.0%) - 3 jugadores detectados
  Frame 50/481 (10.4%) - 4 jugadores detectados
  Frame 100/481 (20.8%) - 4 jugadores detectados
  Frame 150/481 (31.2%) - 4 jugadores detectados
  Frame 200/481 (41.6%) - 4 jugadores detectados
  Frame 250/481 (52.0%) - 4 jugadores detectados
  Frame 300/481 (62.4%) - 4 jugadores detectados
  Frame 350/481 (72.8%) - 4 jugadores detectados
  Frame 400/481 (83.2%) - 4 jugadores detectados
  Frame 450/481 (93.6%) - 5 jugadores detectados

✓ Tracking completado: 481 frames procesados
  Total detecciones: 1975
  Promedio por frame: 4.1
  IDs únicos detectados: 33
  IDs: [2, 3, 4, 5, 21, 22, 24, 26, 27, 36, 43, 53, 60, 67, 68, 69, 71, 72, 78, 79, 81, 89, 99, 103, 106, 108, 129, 130, 133, 137, 143, 149, 160]


## 7. Análisis de Tracking

Verificamos la calidad del tracking y detectamos posibles problemas.

In [10]:
# Analizar duración de cada track
track_durations = defaultdict(int)

for frame_idx, detections in tracking_data.items():
    for det in detections:
        track_id = det[0]
        track_durations[track_id] += 1

print("Duración de cada track (en frames):")
for track_id in sorted(track_durations.keys()):
    duration = track_durations[track_id]
    duration_sec = duration / fps
    print(f"  ID {track_id}: {duration} frames ({duration_sec:.1f}s)")

# Detectar frames con oclusiones potenciales (0 o 1 jugador detectado)
frames_with_issues = []
for frame_idx, detections in tracking_data.items():
    if len(detections) <= 1:
        frames_with_issues.append((frame_idx, len(detections)))

if frames_with_issues:
    print(f"\n⚠ {len(frames_with_issues)} frames con posibles oclusiones (≤1 jugador):")
    print(f"  Ejemplos: {frames_with_issues[:5]}")
else:
    print("\n✓ No se detectaron problemas de oclusión significativos")

Duración de cada track (en frames):
  ID 2: 175 frames (5.8s)
  ID 3: 256 frames (8.5s)
  ID 4: 73 frames (2.4s)
  ID 5: 449 frames (15.0s)
  ID 21: 3 frames (0.1s)
  ID 22: 205 frames (6.8s)
  ID 24: 4 frames (0.1s)
  ID 26: 3 frames (0.1s)
  ID 27: 4 frames (0.1s)
  ID 36: 21 frames (0.7s)
  ID 43: 28 frames (0.9s)
  ID 53: 9 frames (0.3s)
  ID 60: 6 frames (0.2s)
  ID 67: 9 frames (0.3s)
  ID 68: 15 frames (0.5s)
  ID 69: 44 frames (1.5s)
  ID 71: 1 frames (0.0s)
  ID 72: 12 frames (0.4s)
  ID 78: 127 frames (4.2s)
  ID 79: 41 frames (1.4s)
  ID 81: 162 frames (5.4s)
  ID 89: 24 frames (0.8s)
  ID 99: 150 frames (5.0s)
  ID 103: 4 frames (0.1s)
  ID 106: 1 frames (0.0s)
  ID 108: 13 frames (0.4s)
  ID 129: 58 frames (1.9s)
  ID 130: 59 frames (2.0s)
  ID 133: 1 frames (0.0s)
  ID 137: 6 frames (0.2s)
  ID 143: 2 frames (0.1s)
  ID 149: 8 frames (0.3s)
  ID 160: 2 frames (0.1s)

⚠ 6 frames con posibles oclusiones (≤1 jugador):
  Ejemplos: [(475, 0), (476, 0), (477, 0), (478, 0), (479

## 7.1. Corrección de IDs Duplicados

Fusionamos tracks que probablemente pertenecen al mismo jugador.
Esto ocurre cuando el tracker pierde temporalmente a un jugador (por oclusión, etc.) y le asigna un nuevo ID al reaparecer.

In [11]:
def get_track_info(tracking_data, track_id):
    """
    Obtiene información de un track: primer frame, último frame, y posiciones.
    """
    frames = []
    positions = []
    
    for frame_idx, detections in tracking_data.items():
        for det in detections:
            if det[0] == track_id:
                frames.append(frame_idx)
                positions.append((det[5], det[6]))  # cx, cy
    
    if not frames:
        return None, None, []
    
    return min(frames), max(frames), positions


def should_merge_tracks(tracking_data, id1, id2, max_frame_gap=30, max_distance=100):
    """
    Determina si dos tracks deberían fusionarse.
    
    Dos tipos de fusión:
    1. Secuencial: id1 termina antes de que id2 empiece (track perdido y recuperado)
    2. Solapamiento: ambos tracks coexisten pero están muy cerca (IDs duplicados)
    
    Args:
        tracking_data: diccionario con datos de tracking
        id1, id2: IDs de los tracks a comparar
        max_frame_gap: máxima brecha temporal en frames
        max_distance: máxima distancia espacial en píxeles
    
    Returns:
        True si los tracks deberían fusionarse
    """
    first1, last1, positions1 = get_track_info(tracking_data, id1)
    first2, last2, positions2 = get_track_info(tracking_data, id2)
    
    if first1 is None or first2 is None:
        return False
    
    # CASO 1: Fusión secuencial (track perdido y recuperado)
    # id1 termina antes de que id2 empiece
    if last1 < first2:
        frame_gap = first2 - last1
        if frame_gap <= max_frame_gap:
            # Distancia espacial entre última posición de id1 y primera de id2
            last_pos1 = positions1[-1]
            first_pos2 = positions2[0]
            
            distance = np.sqrt((last_pos1[0] - first_pos2[0])**2 + 
                              (last_pos1[1] - first_pos2[1])**2)
            
            if distance <= max_distance:
                return True
    
    # CASO 2: Solapamiento temporal (IDs duplicados del mismo jugador)
    # Los tracks coexisten en el tiempo
    overlap_start = max(first1, first2)
    overlap_end = min(last1, last2)
    
    if overlap_start <= overlap_end:
        # Hay solapamiento temporal, verificar proximidad espacial en frames comunes
        distances = []
        
        for frame_idx in range(overlap_start, overlap_end + 1):
            if frame_idx in tracking_data:
                pos1 = None
                pos2 = None
                
                for det in tracking_data[frame_idx]:
                    if det[0] == id1:
                        pos1 = (det[5], det[6])  # cx, cy
                    if det[0] == id2:
                        pos2 = (det[5], det[6])
                
                # Si ambos IDs están presentes en este frame
                if pos1 is not None and pos2 is not None:
                    dist = np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
                    distances.append(dist)
        
        # Si tienen frames en común y están muy cerca, son duplicados
        if distances:
            avg_distance = np.mean(distances)
            # Umbral más estricto para solapamientos (mismo jugador detectado 2 veces)
            if avg_distance <= max_distance * 0.6:  # 60% del umbral normal
                return True
    
    return False


def merge_track_ids(tracking_data, id_from, id_to):
    """
    Fusiona id_from en id_to, reemplazando todas las apariciones.
    """
    for frame_idx in tracking_data:
        detections = tracking_data[frame_idx]
        new_detections = []
        
        for det in detections:
            track_id = det[0]
            if track_id == id_from:
                # Cambiar el ID
                new_det = (id_to,) + det[1:]
                new_detections.append(new_det)
            else:
                new_detections.append(det)
        
        tracking_data[frame_idx] = new_detections


# Obtener todos los IDs únicos
all_ids = sorted(all_track_ids)

print("Analizando posibles fusiones de IDs...")
print(f"IDs originales: {all_ids}\n")

# Mapa de fusiones: id_original -> id_final
merge_map = {id: id for id in all_ids}

# Comparar todos los pares de IDs con parámetros progresivamente más permisivos
# Esperamos 4 jugadores únicos en el campo
EXPECTED_PLAYERS = 4

# Parámetros iniciales
MAX_FRAME_GAP = 30  # frames (aproximadamente 1 segundo a 30fps)
MAX_DISTANCE = 250  # píxeles

merges_done = []

# Iteración múltiple con estrategia adaptativa
for iteration in range(5):  # Hasta 5 iteraciones
    print(f"\n{'='*60}")
    print(f"ITERACIÓN {iteration + 1}")
    print(f"{'='*60}")
    print(f"  Parámetros: gap={MAX_FRAME_GAP}f, distancia={MAX_DISTANCE}px")
    
    # Recalcular IDs únicos actuales
    current_ids = set()
    for dets in tracking_data.values():
        for det in dets:
            current_ids.add(det[0])
    
    # Analizar frames con exceso de jugadores
    frames_over_limit = []
    for frame_idx, dets in tracking_data.items():
        if len(dets) > EXPECTED_PLAYERS:
            frames_over_limit.append((frame_idx, len(dets)))
    
    print(f"  IDs actuales: {sorted(current_ids)} (total: {len(current_ids)})")
    print(f"  Frames con >{EXPECTED_PLAYERS} jugadores: {len(frames_over_limit)}")
    
    # Si ya tenemos el número esperado de jugadores Y pocos frames problemáticos
    if len(current_ids) <= EXPECTED_PLAYERS:
        print(f"  ✓ Objetivo alcanzado: {len(current_ids)} jugadores")
        break
    
    merges_in_iteration = 0
    all_ids = sorted(current_ids)
    
    # Estrategia: priorizar fusiones que más reduzcan frames problemáticos
    merge_candidates = []
    
    for i, id1 in enumerate(all_ids):
        for id2 in all_ids[i+1:]:
            # Obtener los IDs actuales (considerando fusiones previas)
            current_id1 = merge_map.get(id1, id1)
            current_id2 = merge_map.get(id2, id2)
            
            if current_id1 == current_id2:
                continue  # Ya fusionados
            
            # Verificar si deberían fusionarse (bidireccional)
            should_merge = should_merge_tracks(tracking_data, current_id1, current_id2, 
                                               MAX_FRAME_GAP, MAX_DISTANCE)
            
            if should_merge:
                # Calcular cuántos frames problemáticos se resolverían
                frames_fixed = 0
                for frame_idx, dets in tracking_data.items():
                    ids_in_frame = [d[0] for d in dets]
                    if current_id1 in ids_in_frame and current_id2 in ids_in_frame:
                        frames_fixed += 1
                
                merge_candidates.append((current_id1, current_id2, frames_fixed))
    
    # Ordenar por impacto (más frames problemáticos resueltos primero)
    merge_candidates.sort(key=lambda x: x[2], reverse=True)
    
    print(f"  Candidatos a fusión: {len(merge_candidates)}")
    
    # Realizar fusiones
    for current_id1, current_id2, frames_fixed in merge_candidates:
        # Verificar que aún no han sido fusionados
        if merge_map.get(current_id1, current_id1) == merge_map.get(current_id2, current_id2):
            continue
        
        merge_from = max(current_id1, current_id2)
        merge_to = min(current_id1, current_id2)
        
        print(f"    Fusionando ID {merge_from} → ID {merge_to} (resuelve {frames_fixed} frames)")
        merge_track_ids(tracking_data, merge_from, merge_to)
        
        # Actualizar mapa de fusiones
        for key in merge_map:
            if merge_map[key] == merge_from:
                merge_map[key] = merge_to
        merge_map[merge_from] = merge_to
        
        merges_done.append((merge_from, merge_to))
        merges_in_iteration += 1
    
    print(f"  Fusiones realizadas: {merges_in_iteration}")
    
    # Si no hubo fusiones, aumentar tolerancia para siguiente iteración
    if merges_in_iteration == 0:
        MAX_FRAME_GAP = int(MAX_FRAME_GAP * 1.5)
        MAX_DISTANCE = int(MAX_DISTANCE * 1.4)
        print(f"  Aumentando tolerancia para próxima iteración...")

# Recalcular IDs únicos después de las fusiones
all_track_ids_merged = set()
for dets in tracking_data.values():
    for det in dets:
        all_track_ids_merged.add(det[0])

print(f"\n{'='*60}")
print(f"✓ FUSIONES COMPLETADAS")
print(f"{'='*60}")
print(f"  Total de fusiones: {len(merges_done)}")
print(f"  IDs originales: {sorted(all_track_ids)}")
print(f"  IDs finales: {sorted(all_track_ids_merged)}")
print(f"  Reducción: {len(all_track_ids)} → {len(all_track_ids_merged)} IDs")

# Análisis de frames con múltiples jugadores
frames_distribution = defaultdict(int)
for frame_idx, dets in tracking_data.items():
    num_players = len(dets)
    frames_distribution[num_players] += 1

print(f"\n📊 Distribución de jugadores por frame:")
for num_players in sorted(frames_distribution.keys()):
    count = frames_distribution[num_players]
    percentage = (count / len(tracking_data)) * 100
    bar = "█" * int(percentage / 2)
    print(f"  {num_players} jugadores: {count:4d} frames ({percentage:5.1f}%) {bar}")

# Análisis de calidad
frames_over_4 = sum(count for num, count in frames_distribution.items() if num > EXPECTED_PLAYERS)
frames_under_4 = sum(count for num, count in frames_distribution.items() if num < EXPECTED_PLAYERS)
frames_exactly_4 = frames_distribution.get(EXPECTED_PLAYERS, 0)

print(f"\n📈 Calidad del tracking:")
print(f"  Frames con exactamente {EXPECTED_PLAYERS} jugadores: {frames_exactly_4} ({frames_exactly_4/len(tracking_data)*100:.1f}%)")
print(f"  Frames con más de {EXPECTED_PLAYERS} jugadores: {frames_over_4} ({frames_over_4/len(tracking_data)*100:.1f}%)")
print(f"  Frames con menos de {EXPECTED_PLAYERS} jugadores: {frames_under_4} ({frames_under_4/len(tracking_data)*100:.1f}%)")

if len(all_track_ids_merged) > EXPECTED_PLAYERS:
    print(f"\n⚠ ADVERTENCIA: {len(all_track_ids_merged)} IDs únicos (esperados: {EXPECTED_PLAYERS})")
    print(f"  Tip: Ejecuta esta celda nuevamente o aumenta MAX_DISTANCE inicial")
elif len(all_track_ids_merged) == EXPECTED_PLAYERS:
    print(f"\n✅ PERFECTO: Exactamente {EXPECTED_PLAYERS} jugadores detectados")
else:
    print(f"\n⚠ Solo {len(all_track_ids_merged)} jugadores (esperados: {EXPECTED_PLAYERS})")

# Actualizar variable global
all_track_ids = all_track_ids_merged

Analizando posibles fusiones de IDs...
IDs originales: [2, 3, 4, 5, 21, 22, 24, 26, 27, 36, 43, 53, 60, 67, 68, 69, 71, 72, 78, 79, 81, 89, 99, 103, 106, 108, 129, 130, 133, 137, 143, 149, 160]


ITERACIÓN 1
  Parámetros: gap=30f, distancia=250px
  IDs actuales: [2, 3, 4, 5, 21, 22, 24, 26, 27, 36, 43, 53, 60, 67, 68, 69, 71, 72, 78, 79, 81, 89, 99, 103, 106, 108, 129, 130, 133, 137, 143, 149, 160] (total: 33)
  Frames con >4 jugadores: 135
  Candidatos a fusión: 61
    Fusionando ID 4 → ID 3 (resuelve 73 frames)
    Fusionando ID 130 → ID 5 (resuelve 58 frames)
    Fusionando ID 129 → ID 81 (resuelve 23 frames)
    Fusionando ID 43 → ID 22 (resuelve 20 frames)
    Fusionando ID 36 → ID 2 (resuelve 16 frames)
    Fusionando ID 22 → ID 4 (resuelve 15 frames)
    Fusionando ID 68 → ID 22 (resuelve 14 frames)
    Fusionando ID 69 → ID 2 (resuelve 10 frames)
    Fusionando ID 79 → ID 3 (resuelve 10 frames)
    Fusionando ID 89 → ID 5 (resuelve 10 frames)
    Fusionando ID 130 → ID 78 (resu

## 7.2. Análisis Final del Tracking Corregido

Verificamos la calidad del tracking después de las correcciones.

In [12]:
# Recalcular duración de cada track después de fusiones
track_durations_final = defaultdict(int)

for frame_idx, detections in tracking_data.items():
    for det in detections:
        track_id = det[0]
        track_durations_final[track_id] += 1

print("Duración de cada track DESPUÉS de corrección (en frames):")
for track_id in sorted(track_durations_final.keys()):
    duration = track_durations_final[track_id]
    duration_sec = duration / fps
    percentage = (duration / total_frames) * 100
    print(f"  ID {track_id}: {duration} frames ({duration_sec:.1f}s, {percentage:.1f}% del video)")

# Estadísticas mejoradas
print(f"\n📊 Resumen del tracking corregido:")
print(f"  • Jugadores únicos: {len(all_track_ids_merged)}")
print(f"  • Frame más poblado: {max(len(dets) for dets in tracking_data.values())} jugadores")
print(f"  • Frame menos poblado: {min(len(dets) for dets in tracking_data.values())} jugadores")

# Verificar continuidad
print(f"\n🔍 Verificación de continuidad:")
for track_id in sorted(all_track_ids_merged):
    first_frame, last_frame, positions = get_track_info(tracking_data, track_id)
    expected_frames = last_frame - first_frame + 1
    actual_frames = len(positions)
    continuity = (actual_frames / expected_frames) * 100
    
    if continuity < 80:
        print(f"  ⚠ ID {track_id}: {continuity:.1f}% continuidad (gaps detectados)")
    else:
        print(f"  ✓ ID {track_id}: {continuity:.1f}% continuidad")

Duración de cada track DESPUÉS de corrección (en frames):
  ID 2: 250 frames (8.3s, 52.0% del video)
  ID 3: 401 frames (13.4s, 83.4% del video)
  ID 4: 375 frames (12.5s, 78.0% del video)
  ID 5: 565 frames (18.8s, 117.5% del video)
  ID 69: 233 frames (7.8s, 48.4% del video)
  ID 78: 1 frames (0.0s, 0.2% del video)
  ID 79: 150 frames (5.0s, 31.2% del video)

📊 Resumen del tracking corregido:
  • Jugadores únicos: 7
  • Frame más poblado: 7 jugadores
  • Frame menos poblado: 0 jugadores

🔍 Verificación de continuidad:
  ✓ ID 2: 108.7% continuidad
  ✓ ID 3: 90.7% continuidad
  ✓ ID 4: 98.4% continuidad
  ✓ ID 5: 118.9% continuidad
  ✓ ID 69: 114.8% continuidad
  ✓ ID 78: 100.0% continuidad
  ✓ ID 79: 96.8% continuidad


## 7.3. Exportar Datos de Tracking a CSV

Guardamos todas las posiciones de los jugadores en un archivo CSV para análisis posterior y aplicación de homografía.

In [ ]:
import csv
import json

# Preparar datos para exportación
tracking_list = []

for frame_idx, detections in sorted(tracking_data.items()):
    for det in detections:
        track_id, x1, y1, x2, y2, cx, cy = det
        tracking_list.append({
            'frame': frame_idx,
            'track_id': track_id,
            'bbox_x1': x1,
            'bbox_y1': y1,
            'bbox_x2': x2,
            'bbox_y2': y2,
            'center_x': cx,
            'center_y': cy,
            'timestamp_sec': frame_idx / fps
        })

# Exportar a CSV
csv_filename = "tracking_data.csv"
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['frame', 'track_id', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 
                  'center_x', 'center_y', 'timestamp_sec']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    writer.writerows(tracking_list)

print(f"✓ Datos exportados a '{csv_filename}'")
print(f"  Total de registros: {len(tracking_list)}")
print(f"  Columnas: frame, track_id, bbox (x1,y1,x2,y2), center (x,y), timestamp")

# También exportar resumen en JSON
summary_data = {
    'video_info': {
        'fps': fps,
        'total_frames': total_frames,
        'width': width,
        'height': height
    },
    'campo_puntos': puntos_campo.tolist(),
    'jugadores_ids': sorted(all_track_ids_merged),
    'num_jugadores': len(all_track_ids_merged),
    'total_detecciones': len(tracking_list)
}

json_filename = "tracking_summary.json"
with open(json_filename, 'w', encoding='utf-8') as jsonfile:
    json.dump(summary_data, jsonfile, indent=2)

print(f"\n✓ Resumen exportado a '{json_filename}'")
print(f"  Incluye: info del video, puntos del campo, IDs de jugadores")

# Crear CSV por jugador (opcional, más fácil para análisis individual)
for track_id in sorted(all_track_ids_merged):
    player_data = [d for d in tracking_list if d['track_id'] == track_id]
    player_filename = f"tracking_player_{track_id}.csv"
    
    with open(player_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(player_data)
    
    print(f"  - '{player_filename}': {len(player_data)} posiciones")

print(f"\n📁 Archivos generados en: {os.getcwd()}")

## 8. Visualización del Resultado

Mostramos el video con las detecciones y tracking aplicados.
- **Cuadros verdes**: bounding boxes de cada jugador
- **Punto verde**: posición de referencia (pies)
- **ID**: identificador único de cada jugador
- **Polígono amarillo**: área del campo con margen

Presiona **ESC** para salir.

In [13]:
# Reiniciar video
video.set(cv2.CAP_PROP_POS_FRAMES, 0)
frame_idx = 0

# Calcular polígono expandido para visualización
center = np.mean(puntos_campo, axis=0)
max_y = np.max(puntos_campo[:, 1])
expanded_polygon = []

for pt in puntos_campo:
    if abs(pt[1] - max_y) < 5:
        direction = pt - center
        direction[1] = min(0, direction[1])
        expanded_pt = pt + direction * MARGIN_PERCENT
    else:
        direction = pt - center
        expanded_pt = pt + direction * MARGIN_PERCENT
    expanded_polygon.append(expanded_pt)

expanded_polygon = np.array(expanded_polygon, dtype=np.int32)

print("Reproduciendo video con tracking...")
print("Presiona ESC para salir\n")

cv2.namedWindow("Tracking Offline - Jugadores", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Tracking Offline - Jugadores", 1200, 800)

while True:
    ret, frame = video.read()
    if not ret:
        # Reiniciar video (loop)
        video.set(cv2.CAP_PROP_POS_FRAMES, 0)
        frame_idx = 0
        continue
    
    # Dibujar polígono del campo expandido
    cv2.polylines(frame, [expanded_polygon], True, (0, 255, 255), 2)
    
    # Obtener detecciones del frame actual
    if frame_idx in tracking_data:
        detections = tracking_data[frame_idx]
        
        for det in detections:
            track_id, x1, y1, x2, y2, cx, cy = det
            
            # Dibujar bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # Dibujar punto de referencia (pies)
            cv2.circle(frame, (cx, cy), 6, (0, 255, 0), -1)
            
            # Dibujar ID
            cv2.putText(frame, f"ID {track_id}", (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    # Información del frame
    info_text = f"Frame: {frame_idx}/{total_frames} | Jugadores: {len(tracking_data.get(frame_idx, []))}"
    cv2.putText(frame, info_text, (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    cv2.imshow("Tracking Offline - Jugadores", frame)
    
    # Control de reproducción
    key = cv2.waitKey(30) & 0xFF
    if key == 27:  # ESC
        break
    
    frame_idx += 1

video.release()
cv2.destroyAllWindows()
print("✓ Visualización finalizada")

Reproduciendo video con tracking...
Presiona ESC para salir

✓ Visualización finalizada


## Resumen

Este notebook ha implementado:
1. ✓ Detección del campo de juego mediante selección manual de puntos
2. ✓ Tracking offline de jugadores usando YOLOv11
3. ✓ Filtrado de detecciones dentro del área del campo (con margen del 10%)
4. ✓ Manejo de oclusiones mediante tracking persistente
5. ✓ Visualización de resultados en tiempo real

**Ventajas del tracking offline:**
- Mayor precisión al procesar todo el video de una vez
- Mejor manejo de oclusiones temporales
- Posibilidad de análisis posteriores (trayectorias, estadísticas, etc.)

**Próximos pasos posibles:**
- Exportar trayectorias de los jugadores
- Análisis de patrones de movimiento
- Detección de eventos (saques, remates, etc.)
- Proyección en el mapa del campo